# Session 2 - Data Preparation, Feature Engineering and Data Leakage

**Course:** Machine Learning in Geoscience - A Project-Based Course  
**Project:** Wind-speed forecasting at Abali  
**Session length:** 90 minutes  
**Data source (the only one used):** `data/raw/abali.xlsx`

---

## Learning objectives

By the end of this session you will be able to:

1. Turn the clean hourly series into a **leakage-safe modelling table** for a chosen forecast horizon.
2. Build and justify four families of features: current observations, lags, rolling windows, and calendar/cyclical encodings.
3. Define the target for a forecast horizon correctly, and state the **forecast origin** convention out loud.
4. Recognise the four ways leakage enters a time-series project - and demonstrate each one empirically.
5. Produce a chronological train/validation/test split with a purge gap, and explain why shuffling is fatal here.
6. Fit any transformation (scaling, feature selection) on the **training period only**.
7. Deliver the final modelling dataset for Abali, in memory, for `H = 3, 6, 12, 24` h.

## Prerequisites

Session 1 (or equivalent familiarity with the Abali record: 10-minute raw file, hourly grid, gaps, sentinels).

---

## 1. Setup

Everything in this notebook is rebuilt in memory from the raw Excel file, exactly as in Session 1. Nothing is written to disk.

Run this cell block first; it defines the three helpers used by Sessions 2-7.

| Helper | Purpose |
|---|---|
| `load_abali_hourly()` | raw Excel -> clean hourly series on a regular grid |
| `build_horizon_dataset(hourly, H)` | hourly series -> leakage-safe `X`, `y` for horizon `H` |
| `chronological_split(n, gap=H)` | indices of train / validation / test with a purge gap |

In [1]:
# ---------------------------------------------------------------------------
# Session 2 setup: libraries, options, helpers (self-contained)
# ---------------------------------------------------------------------------
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 80)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 4)

# toggles for a 90-minute classroom session
QUICK_MODE = True        # True -> smaller/faster models in the demonstration cells

HAS_SNS = False
try:
    import seaborn as sns
    HAS_SNS = True
except Exception as exc:
    print('[optional] seaborn not available:', exc)

# Data paths. The primary candidate is relative to the repository root; the
# alternatives keep the notebook working when Jupyter starts the kernel inside
# course/notebooks/ (JupyterLab does this by default). No absolute paths.
RAW_CANDIDATES = ['data/raw/abali.xlsx', '../data/raw/abali.xlsx', '../../data/raw/abali.xlsx']


def resolve_raw_path(candidates=RAW_CANDIDATES):
    '''Return the first existing relative path to the Abali raw data file.'''
    for candidate in candidates:
        if os.path.exists(candidate):
            print('using raw data file:', candidate)
            return candidate
    raise FileNotFoundError(
        'Could not find abali.xlsx. Expected at ' + candidates[0] +
        ' relative to the repository root (the folder holding data/, src/, course/).'
    )


RAW_PATH = resolve_raw_path()
COL_MAP = {
    'DateTimes': 'datetime',
    'Wind Speed(m/s)': 'wind_speed_ms',
    'Wind Direction(D)': 'wind_direction_deg',
    'Air Pressure(hPa)': 'pressure_hpa',
    'Air Temperature(C)': 'temperature_c',
    'rel. Humidity(%)': 'humidity',
}
DATA_COLS = ['wind_speed_ms', 'wind_direction_deg', 'pressure_hpa', 'temperature_c', 'humidity']
PHYSICAL_BOUNDS = {
    'wind_speed_ms': (0.0, 60.0),
    'wind_direction_deg': (0.0, 360.0),
    'pressure_hpa': (500.0, 1100.0),
    'temperature_c': (-60.0, 60.0),
    'humidity': (0.0, 100.0),
}
HORIZONS = [3, 6, 12, 24]

print('Python:', sys.version.split()[0], '| pandas:', pd.__version__, '| numpy:', np.__version__)
print('QUICK_MODE:', QUICK_MODE)

using raw data file: ../../data/raw/abali.xlsx
Python: 3.13.7 | pandas: 2.3.2 | numpy: 2.5.2
QUICK_MODE: True


In [2]:
def load_abali_hourly(path=RAW_PATH, verbose=True):
    """Raw Abali Excel -> clean HOURLY DataFrame (in memory, nothing written).

    1. read the 10-minute export
    2. keep + rename six canonical columns
    3. sentinels (<= -900) and physically impossible values -> NaN
    4. drop unusable / duplicate timestamps
    5. average the 10-minute records to hourly means
    6. reindex onto a perfect hourly grid
    7. interpolate short gaps (<= 2 h) of AUXILIARY variables only

    Returns a DataFrame indexed by datetime with DATA_COLS + ['n_obs'].
    """
    if not os.path.exists(path):
        raise FileNotFoundError('Could not find ' + path + ' - run from the repository root.')
    raw = pd.read_excel(path)[list(COL_MAP)].rename(columns=COL_MAP)
    raw['datetime'] = pd.to_datetime(raw['datetime'], errors='coerce')
    for c in DATA_COLS:
        raw[c] = pd.to_numeric(raw[c], errors='coerce')
    raw[DATA_COLS] = raw[DATA_COLS].mask(raw[DATA_COLS] <= -900)
    for c, (lo, hi) in PHYSICAL_BOUNDS.items():
        raw[c] = raw[c].mask((raw[c] < lo) | (raw[c] > hi))
    raw = raw.dropna(subset=['datetime'])
    raw = raw.sort_values('datetime').drop_duplicates(subset='datetime', keep='first')

    g = raw.set_index('datetime')
    hourly = g[DATA_COLS].resample('1h').mean()
    n_obs = g['wind_speed_ms'].resample('1h').count()
    grid = pd.date_range(hourly.index.min(), hourly.index.max(), freq='h')
    hourly = hourly.reindex(grid)
    hourly['n_obs'] = n_obs.reindex(grid).fillna(0).astype(int)
    hourly.index.name = 'datetime'

    aux = [c for c in DATA_COLS if c != 'wind_speed_ms']
    hourly[aux] = hourly[aux].interpolate(method='linear', limit=2, limit_area='inside')

    if verbose:
        print('hourly rows:', len(hourly), '| range:', hourly.index.min(), '->', hourly.index.max())
        print('missing hours per variable:')
        print(hourly[DATA_COLS].isna().sum().to_string())
    return hourly


hourly = load_abali_hourly()

hourly rows: 10440 | range: 2024-01-01 00:00:00 -> 2025-03-10 23:00:00
missing hours per variable:
wind_speed_ms         111
wind_direction_deg     94
pressure_hpa           94
temperature_c          94
humidity              442


---

## 2. The one rule that governs everything

> **At every forecast origin `T`, a feature may only use information that was available at or before `T`.**

The model is trained to predict the wind speed at `T + H`:

```
            known at T                          unknown
  <---------------------------------|---------------------------------->
  observations, lags, rolling stats |   y  =  wind speed at T + H
                                    T                                T+H
```

Every design decision in this notebook is just an application of that sentence.

### 2.1 What does a feature look like on the hourly grid?

| Family | Example | Available at `T`? |
|---|---|---|
| current observation | `wind_speed_ms` at `T` | yes |
| lag | `wind_speed_ms` at `T-1`, `T-6`, `T-24` | yes |
| rolling window | mean/max/std of the **past** 3, 6, 24 h ending at `T-1` | yes |
| calendar | hour of day, day of year, cyclical `sin`/`cos` | yes - computable for any future timestamp |
| **target** | `wind_speed_ms` at `T+H` | **no - this is what we predict** |

The rolling window deserves special attention. `df['x'].rolling(24).mean()` computed at row `T` includes row `T` itself. Including the *current* hour is legitimate (it is observed), but including any **shift(-k)** version of an input would reach into the future. We standardise on the safest pattern:

```python
past = series.shift(1)              # strictly the past
feature = past.rolling(24).mean()   # window = T-24 ... T-1
```

Being *strictly* past-only costs almost nothing in accuracy and removes an entire class of arguments - and you never have to prove again that a feature was available.

### 2.2 Data-quality review before feature building

Missing values, outliers and continuity have different treatments depending on the **role** of the variable:

| Situation | Target (wind speed) | Auxiliary input |
|---|---|---|
| Isolated missing hour | leave `NaN`, never fabricate | linear interpolation over <= 2 h |
| Long outage (days) | drop those rows | drop those rows |
| Physically impossible value | set to `NaN` | set to `NaN` |
| Extreme but plausible wind | **keep** - it is the phenomenon of interest | keep |
| Stuck sensor (long constant run) | flag and discuss; do not silently delete | flag |

The last row is the one students get wrong most often. A 17 m/s hour is not an outlier to be cleaned away: it is the *event* the forecast exists to predict. Removing it would make every metric look better and the model useless.

In [3]:
# Data-quality review on the clean hourly series.
ws = hourly['wind_speed_ms']

summary = pd.DataFrame({
    'missing_hours': hourly[DATA_COLS].isna().sum(),
    'missing_pct': 100 * hourly[DATA_COLS].isna().mean(),
    'min': hourly[DATA_COLS].min(),
    'max': hourly[DATA_COLS].max(),
    'std': hourly[DATA_COLS].std(),
})
print(summary.round(3).to_string())

# Outlier screening with the IQR rule, as a DIAGNOSTIC only.
q1, q3 = ws.quantile(0.25), ws.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
n_above = int((ws > upper).sum())
print()
print('IQR fence for wind speed : {:.3f} m/s (Q3 + 1.5 IQR)'.format(upper))
print('hours above the fence    :', n_above, '({:.2%} of all hours)'.format(n_above / len(ws)))
print('max hourly wind speed    : {:.2f} m/s'.format(ws.max()))
print()
print('DECISION: these hours are strong-wind events, not measurement errors.')
print('They stay in the dataset. In Session 6 we will measure the error of our')
print('models separately for calm, moderate and strong-wind hours, which is the')
print('scientifically useful way to treat them.')

# A genuinely suspicious pattern: long runs of identical values (possible stuck sensor).
zero_runs = ws.eq(0)
runs = (zero_runs != zero_runs.shift()).cumsum()
longest_zero_run = int(zero_runs.groupby(runs).sum().max())
print()
print('longest run of consecutive exactly-zero hours:', longest_zero_run)
print('interpretation: a multi-day run of exact zeros would indicate a stuck')
print('anemometer; a few hours is ordinary calm air at night.')

                    missing_hours  missing_pct      min      max     std
wind_speed_ms                 111        1.063    0.000   18.080   2.379
wind_direction_deg             94        0.900    7.333  355.833  61.598
pressure_hpa                   94        0.900  744.950  765.983   3.427
temperature_c                  94        0.900  -17.217   32.317  10.159
humidity                      442        4.234    8.000  100.000  23.100

IQR fence for wind speed : 6.925 m/s (Q3 + 1.5 IQR)
hours above the fence    : 518 (4.96% of all hours)
max hourly wind speed    : 18.08 m/s

DECISION: these hours are strong-wind events, not measurement errors.
They stay in the dataset. In Session 6 we will measure the error of our
models separately for calm, moderate and strong-wind hours, which is the
scientifically useful way to treat them.

longest run of consecutive exactly-zero hours: 9
interpretation: a multi-day run of exact zeros would indicate a stuck
anemometer; a few hours is ordinary calm ai

---

## 3. Building the target

For a **direct** multi-horizon setup we create one table per horizon:

```
y_T = wind_speed at T + H          target, in m/s
```

and we discard the last `H` rows, where `T + H` falls beyond the end of the record.

### Three practical consequences

1. **Different horizons = different datasets.** The number of usable rows shrinks as `H` grows, and the target distribution changes, because a 24 h-ahead target is a different random variable from a 3 h-ahead target.
2. **The forecast origin is part of the definition.** Always write the sentence: *features known at T, target at T+H.*
3. **Purge gaps.** If the training row at `T` has a target at `T+H`, and validation starts at `T+1`, the training target and the validation feature window overlap in time. We will remove `H` rows at each boundary - the *purge* - so that no validation information sits inside a training target window.

### Why cyclical encoding?

Raw `hour` is a poor feature: a linear model sees 23:00 and 00:00 as maximally different, although they are adjacent. Encoding the cycle on a circle fixes that:

```
hour_sin = sin(2 * pi * hour / 24)
hour_cos = cos(2 * pi * hour / 24)
```

and identically for the day of the year with a period of 365.25. We keep the raw `hour` and `month` as well, because tree ensembles can split on them directly - with the caveat that `hour` still imposes the artificial 23 -> 0 jump.

In [4]:
def build_horizon_dataset(hourly, horizon, lags=(1, 2, 3, 6, 12, 24), windows=(3, 6, 24)):
    """Build a leakage-safe feature matrix and target for a direct H-hour forecast.

    Contract
    --------
    X row at time T contains ONLY information available at or before T.
    y row at time T is the wind speed at T + horizon.
    Rows with any NaN in a feature or in the target are dropped.

    Feature families
    ----------------
    current    : the observed value at T (legitimate: T is in the past relative to T+H)
    lag        : value at T - k for k in lags
    rolling    : mean/max/std over the strictly past window (T-w ... T-1)
    calendar   : hour, day of year, month, weekday, weekend flag
    cyclical   : sin/cos of hour and of day of year

    Returns
    -------
    X : DataFrame (index = datetime of the forecast origin)
    y : Series    (index = datetime of the forecast origin)
    """
    df = hourly.copy()

    # --- wind direction is circular: encode it before anything else ---------
    rad = np.radians(df['wind_direction_deg'])
    df['wind_dir_sin'] = np.sin(rad)
    df['wind_dir_cos'] = np.cos(rad)
    df = df.drop(columns=['wind_direction_deg', 'n_obs'])   # raw degrees and the QC flag are not features

    base = ['wind_speed_ms', 'pressure_hpa', 'temperature_c', 'humidity', 'wind_dir_sin', 'wind_dir_cos']
    angular = ['wind_dir_sin', 'wind_dir_cos']    # max/std of sin/cos of a direction is meaningless

    # --- target: the value H hours AFTER the forecast origin ----------------
    df['target'] = df['wind_speed_ms'].shift(-horizon)

    features = list(base)                       # (a) current observations at T

    for c in base:                              # (b) lags: value at T-k
        for k in lags:
            name = c + '_lag' + str(k)
            df[name] = df[c].shift(k)
            features.append(name)

    for c in base:                              # (c) rolling statistics of the PAST only
        past = df[c].shift(1)
        for w in windows:
            name = c + '_roll' + str(w) + '_mean'
            df[name] = past.rolling(w).mean()
            features.append(name)
            if c not in angular:
                name = c + '_roll' + str(w) + '_max'
                df[name] = past.rolling(w).max()
                features.append(name)
                name = c + '_roll' + str(w) + '_std'
                df[name] = past.rolling(w).std()
                features.append(name)

    idx = df.index                              # (d) calendar and cyclical features
    df['hour'] = idx.hour
    df['dayofyear'] = idx.dayofyear
    df['month'] = idx.month
    df['dayofweek'] = idx.dayofweek
    df['is_weekend'] = (idx.dayofweek >= 5).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * idx.hour / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * idx.hour / 24.0)
    df['doy_sin'] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * idx.dayofyear / 365.25)
    features += ['hour', 'dayofyear', 'month', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']

    # --- drop warm-up rows (NaN lags) and the last H rows (target unknown) --
    valid = df[features].notna().all(axis=1) & df['target'].notna()
    X = df.loc[valid, features].copy()
    y = df.loc[valid, 'target'].copy()
    X.index.name = 'origin_time'
    return X, y


X6, y6 = build_horizon_dataset(hourly, 6)
print('H = 6 h -> feature matrix', X6.shape)
print('target (first 3 rows):')
print(y6.head(3).to_string())
print()
print('verification that y is really the value 6 hours after the origin:')
for origin in [X6.index[500], X6.index[5000]]:
    print('  X at {} -> y = {:.3f} ; hourly at {} = {:.3f}'.format(
        origin, y6.loc[origin], origin + pd.Timedelta(hours=6), hourly['wind_speed_ms'].loc[origin + pd.Timedelta(hours=6)]))

H = 6 h -> feature matrix (8605, 93)
target (first 3 rows):
datetime
2024-01-02 00:00:00    1.166667
2024-01-02 01:00:00    2.300000
2024-01-02 02:00:00    2.716667

verification that y is really the value 6 hours after the origin:
  X at 2024-02-04 01:00:00 -> y = 2.533 ; hourly at 2024-02-04 07:00:00 = 2.533
  X at 2024-09-07 04:00:00 -> y = 3.733 ; hourly at 2024-09-07 10:00:00 = 3.733


C:\Users\Amir\AppData\Local\Temp\ipykernel_7288\745074744.py:87: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  origin, y6.loc[origin], origin + pd.Timedelta(hours=6), hourly['wind_speed_ms'].loc[origin + pd.Timedelta(hours=6)]))


The verification cell above is not decoration: **always print the target next to the raw series at `T+H` for a couple of rows.** Alignment bugs are invisible in shape checks and catastrophic in results.

**Check for understanding:** the feature `wind_speed_ms` (value at `T`) sits in the same table as `y` (value at `T+H`). Is that leakage? *No - `T` is observed when the forecast is issued. It would only be leakage if we claimed to issue the forecast without current observations, i.e. the NWP-driven setup from Session 1. State your setup explicitly in the paper.*

---

## 4. Data leakage, demonstrated

Rather than defining leakage abstractly, we will *measure* it four times. Each demonstration is short and each one is a mistake that real papers contain.

| # | Leak | What it looks like |
|---|---|---|
| A | target leakage | a feature that encodes the answer |
| B | temporal leakage through shuffling | random train/test split on autocorrelated data |
| C | back-filling | `fillna(method='bfill')` or `interpolate(limit_direction='both')` |
| D | pre-split scaling / feature selection | a scaler or selector fitted on the whole record |

For A and B we need a model; we use a small random forest, deliberately small so that the whole class can run it.

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

N_TREES = 60 if QUICK_MODE else 200
N_JOBS = 2   # joblib spawns one worker process per job: keep this small on a laptop

def rf_mae(Xtr, ytr, Xte, yte, label):
    """Fit a small random forest and return (MAE, R2) on the test part."""
    # max_features='sqrt' is much faster and lighter than the default (all features)
    # and is the standard setting for regression forests on this kind of table.
    model = RandomForestRegressor(n_estimators=N_TREES, max_features='sqrt',
                                  min_samples_leaf=2, random_state=SEED, n_jobs=N_JOBS)
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    mae = mean_absolute_error(yte, pred)
    r2 = r2_score(yte, pred)
    print('  {:<42s} MAE = {:.4f} m/s   R2 = {:+.4f}'.format(label, mae, r2))
    return mae, r2

# Reference: what a legitimate CHRONOLOGICAL split gives us (H = 6).
n = len(y6)
n_train, n_val = int(0.70 * n), int(0.15 * n)
X_tr, y_tr = X6.iloc[:n_train], y6.iloc[:n_train]
X_te, y_te = X6.iloc[n_train + n_val:], y6.iloc[n_train + n_val:]

print('H = 6 h, legitimate chronological split:')
rf_mae(X_tr, y_tr, X_te, y_te, 'chronological split (honest)')

H = 6 h, legitimate chronological split:
  chronological split (honest)               MAE = 1.3584 m/s   R2 = +0.1589


(1.3584369395106526, 0.15892979629077142)

In [6]:
# ---------------------------------------------------------------------------
# LEAK A - target leakage: a feature that carries the answer
# ---------------------------------------------------------------------------
rng = np.random.RandomState(SEED)
X_leak = X6.copy()
# The cheat feature is the target plus tiny noise, exactly as a mis-merged column
# or an off-by-one shift would produce it.
X_leak['wind_speed_next_day'] = y6.values + rng.normal(0, 0.05, len(y6))

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_leak, y6, test_size=0.2, random_state=SEED)
print('H = 6 h, with a leaky feature (random split):')
rf_mae(Xa_tr, ya_tr, Xa_te, ya_te, 'LEAKY feature present')
print()
print('The forest found the cheat column immediately: MAE drops from about 1.36 m/s')
print('on the honest split to about 0.39 here, and R2 jumps from 0.16 to 0.91 - on a')
print('target whose standard deviation is 2.4 m/s. Note that the score is not 1.00:')
print('the cheat column is noisy and the forest is deliberately small, but the jump')
print('is already unmistakable. The signature to remember: a hard problem that')
print('suddenly looks easy, with a top feature that has no physical interpretation.')
print()
print('Feature importance exposes it - the top feature by far is the cheat column:')
rf_all = RandomForestRegressor(n_estimators=N_TREES, max_features='sqrt', min_samples_leaf=2,
                               random_state=SEED, n_jobs=N_JOBS).fit(Xa_tr, ya_tr)
imp = pd.Series(rf_all.feature_importances_, index=X_leak.columns).sort_values(ascending=False)
print(imp.head(5).round(4).to_string())

H = 6 h, with a leaky feature (random split):
  LEAKY feature present                      MAE = 0.3886 m/s   R2 = +0.9077

The forest found the cheat column immediately: MAE drops from about 1.36 m/s
on the honest split to about 0.39 here, and R2 jumps from 0.16 to 0.91 - on a
target whose standard deviation is 2.4 m/s. Note that the score is not 1.00:
the cheat column is noisy and the forest is deliberately small, but the jump
is already unmistakable. The signature to remember: a hard problem that
suddenly looks easy, with a top feature that has no physical interpretation.

Feature importance exposes it - the top feature by far is the cheat column:
wind_speed_next_day         0.3848
wind_speed_ms               0.0327
hour                        0.0309
wind_speed_ms_lag1          0.0295
wind_speed_ms_roll3_mean    0.0238


In [7]:
# ---------------------------------------------------------------------------
# LEAK B - the same honest features, but with a SHUFFLED split
# ---------------------------------------------------------------------------
# This is the default in most machine-learning tutorials: shuffle=True. On
# autocorrelated time series it puts the hours immediately before and after a
# test hour into the training set, so the model can interpolate its way to the
# answer instead of forecasting it.
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X6, y6, test_size=0.2, random_state=SEED, shuffle=True)

print('H = 6 h, same features, two different splits:')
m_honest = rf_mae(X_tr, y_tr, X_te, y_te, 'chronological split')
m_shuf = rf_mae(Xb_tr, yb_tr, Xb_te, yb_te, 'shuffled split (WRONG)')
print()
print('Improvement in MAE caused purely by the split: {:.1f} %'.format(100 * (m_honest[0] - m_shuf[0]) / m_honest[0]))
print('Improvement in R2  caused purely by the split: {:+.3f}'.format(m_shuf[1] - m_honest[1]))
print()
print('Nothing about the model or the features changed. The shuffled split simply')
print('gave the model a look at the neighbouring hours of every test point.')
print('This is the single most common leakage in published weather-ML work.')

H = 6 h, same features, two different splits:
  chronological split                        MAE = 1.3584 m/s   R2 = +0.1589
  shuffled split (WRONG)                     MAE = 0.8303 m/s   R2 = +0.6325

Improvement in MAE caused purely by the split: 38.9 %
Improvement in R2  caused purely by the split: +0.474

Nothing about the model or the features changed. The shuffled split simply
gave the model a look at the neighbouring hours of every test point.
This is the single most common leakage in published weather-ML work.


In [8]:
# ---------------------------------------------------------------------------
# LEAK C - filling missing values with future information
# ---------------------------------------------------------------------------
gap = hourly['humidity'].copy()
# Create an artificial 6-hour outage in the middle of the record to make the
# mechanism visible.
center = len(gap) // 2
gap.iloc[center:center + 6] = np.nan

ffill = gap.ffill()                                   # uses only the past  -> safe
bfill = gap.bfill()                                   # uses the FUTURE      -> leaky
both = gap.interpolate(method='linear', limit_direction='both')   # also reaches forward

window = slice(center - 2, center + 8)
comparison = pd.DataFrame({
    'truth': hourly['humidity'].iloc[window],
    'ffill (safe)': ffill.iloc[window],
    'bfill (leaky)': bfill.iloc[window],
    'interp both (leaky)': both.iloc[window],
})
print('A 6-hour humidity outage in the middle of the record:')
print(comparison.round(2).to_string())
print()
print('At the first missing hour, bfill already knows the value of the hour that')
print('comes AFTER the outage. If that filled value is used as a feature at time T,')
print('the model receives information from the future.')
print()
print('Rule used in this course: forward-fill or backward-looking interpolation only,')
print('and never for the target. In load_abali_hourly we use limit_area=inside with a')
print('2-hour cap, which only ever interpolates BETWEEN two observed hours - and')
print('that is acceptable only because the interpolated variables are not the target.')

A 6-hour humidity outage in the middle of the record:
                     truth  ffill (safe)  bfill (leaky)  interp both (leaky)
datetime                                                                    
2024-08-05 10:00:00  22.50         22.50          22.50                22.50
2024-08-05 11:00:00  23.00         23.00          23.00                23.00
2024-08-05 12:00:00  21.00         23.00          25.83                23.40
2024-08-05 13:00:00  20.33         23.00          25.83                23.81
2024-08-05 14:00:00  20.33         23.00          25.83                24.21
2024-08-05 15:00:00  21.17         23.00          25.83                24.62
2024-08-05 16:00:00  26.17         23.00          25.83                25.02
2024-08-05 17:00:00  27.83         23.00          25.83                25.43
2024-08-05 18:00:00  25.83         25.83          25.83                25.83
2024-08-05 19:00:00  26.17         26.17          26.17                26.17

At the first missing 

In [9]:
# ---------------------------------------------------------------------------
# LEAK D - fitting a transformation before splitting
# ---------------------------------------------------------------------------
from sklearn.preprocessing import StandardScaler, MinMaxScaler

full_scaler = StandardScaler().fit(X6)            # WRONG: sees validation and test
X6_scaled_leaky = pd.DataFrame(full_scaler.transform(X6), index=X6.index, columns=X6.columns)

train_scaler = StandardScaler().fit(X_tr)         # RIGHT: sees the training period only
X_tr_scaled_ok = pd.DataFrame(train_scaler.transform(X_tr), index=X_tr.index, columns=X_tr.columns)
X_te_scaled_ok = pd.DataFrame(train_scaler.transform(X_te), index=X_te.index, columns=X_te.columns)

print('Mean of two features, three ways:')
cols = ['wind_speed_ms', 'wind_speed_ms_roll24_mean']
tab = pd.DataFrame({
    'raw (train part)': X_tr[cols].mean(),
    'scaled with a scaler fitted on the FULL record': X6_scaled_leaky.loc[X_tr.index, cols].mean(),
    'scaled with a scaler fitted on TRAIN only': X_tr_scaled_ok[cols].mean(),
})
print(tab.round(6).to_string())
print()
print('Values from the training period must transform to a mean of exactly 0 when')
print('the scaler was fitted on the training period - and not quite 0 when the scaler')
print('was fitted on everything. The leak is small here because the record is long,')
print('but it is not zero, and on a short record it can be large.')
print()
print('Be honest about the size of the effect: pre-split scaling is usually a small')
print('bias. Pre-split FEATURE SELECTION is a much bigger one, because the selector')
print('learns which columns correlate with the target over the whole record, test')
print('period included.')

# A quantitative demonstration of the bigger version of the same mistake.
corr_full = X6.apply(lambda col: col.corr(y6)).abs()
corr_train = X_tr.apply(lambda col: col.corr(y_tr)).abs()
rank_full = corr_full.sort_values(ascending=False).head(10).index.tolist()
rank_train = corr_train.sort_values(ascending=False).head(10).index.tolist()
print()
print('top-10 features by |correlation with the target|:')
print('  chosen on the FULL record  :', rank_full[:5])
print('  chosen on TRAIN only       :', rank_train[:5])
print('  overlap                    :', len(set(rank_full) & set(rank_train)), 'of 10')

Mean of two features, three ways:
                           raw (train part)  scaled with a scaler fitted on the FULL record  scaled with a scaler fitted on TRAIN only
wind_speed_ms                      2.634261                                        0.087180                                       -0.0
wind_speed_ms_roll24_mean          2.688062                                        0.107658                                        0.0

Values from the training period must transform to a mean of exactly 0 when
the scaler was fitted on the training period - and not quite 0 when the scaler
was fitted on everything. The leak is small here because the record is long,
but it is not zero, and on a short record it can be large.

Be honest about the size of the effect: pre-split scaling is usually a small
bias. Pre-split FEATURE SELECTION is a much bigger one, because the selector
learns which columns correlate with the target over the whole record, test
period included.

top-10 features by |co

### The leakage audit - run this list on your own tables

Go through every feature and ask these questions **in writing**:

1. At which timestamp is this feature's value produced? Is that timestamp `<= T` for every row? (`shift(-k)`, `bfill`, `center=True` rolling windows and `-1` slices all fail here.)
2. Does this feature contain the target for any row, directly or through a rolling window that reaches `T` or later? (The `rolling(...)` default `center=False` is fine; a window built on the *unshifted* series includes `T`, which is still legal, but a window that includes `T+H` is not.)
3. Was any transformation (scaler, imputer, selector, PCA, target encoding) fitted on data that includes the validation or test period?
4. Was the split done by **time** or by random shuffling?
5. Does the split leave `H` rows of overlap between a training target window and a validation feature window? (That is what the purge removes.)
6. Is any identifier (`n_obs`, a row counter, a station code, an index) present as a feature?
7. If you ran a *modern* model (boosting, deep learning) and got `R2 > 0.95` on a hard geophysical target, what is the most likely explanation? *(Almost always a leak. Check before you celebrate.)*

---

## 5. The chronological split

We use three consecutive blocks of time:

| Block | Share | Used for |
|---|---|---|
| train | 70 % | fitting models, fitting scalers, selecting features |
| validation | 15 % | comparing models, tuning hyper-parameters, choosing the final model |
| test | 15 % | **one** final evaluation, touched exactly once (Session 7) |

and a **purge gap of `H` hours** at every boundary, because a training row at `T` carries information about the interval `(T, T+H]`. Without the purge the first `H` validation rows would be partly predicted from training targets.

```
train ......... [purge H] ..... validation .... [purge H] ..... test
targets <= train_end         targets <= val_end
```

### What to check after splitting

1. The blocks do not overlap and are ordered.
2. The gap between blocks is at least `H` hours.
3. The **distributions differ** - and by how much. This dataset has a strong seasonal cycle, so a winter test block is a harder and more honest test than a random sample of the year would be. Report that; do not hide it.

In [10]:
def chronological_split(n, frac=(0.70, 0.15, 0.15), gap=0):
    """Indices of the train / validation / test blocks in chronological order.

    A purge gap of `gap` rows is removed at each boundary so that no training
    target window overlaps the following block.
    """
    n_train = int(frac[0] * n)
    n_val = int(frac[1] * n)
    train = np.arange(0, max(n_train - gap, 1))
    val = np.arange(n_train + gap, max(n_train + n_val - gap, 1))
    test = np.arange(n_train + n_val + gap, n)
    return train, val, test


def split_report(X, y, gap, name=''):
    """Return the three blocks and print a diagnostic table for one horizon."""
    tr, va, te = chronological_split(len(y), gap=gap)
    blocks = {'train': tr, 'validation': va, 'test': te}
    rows = []
    for label, idx in blocks.items():
        rows.append({
            'block': label,
            'n_rows': len(idx),
            'from': X.index[idx[0]],
            'to': X.index[idx[-1]],
            'target_mean': y.iloc[idx].mean(),
            'target_std': y.iloc[idx].std(),
            'target_max': y.iloc[idx].max(),
        })
    tab = pd.DataFrame(rows).set_index('block')
    print('split for ' + name + ' (purge gap = ' + str(gap) + ' h):')
    print(tab.round(3).to_string())
    # integrity checks
    assert tr[-1] < va[0] and va[-1] < te[0], 'blocks are not ordered'
    assert X.index[va[0]] - X.index[tr[-1]] >= pd.Timedelta(hours=gap), 'purge gap too small'
    print('  ordering and purge gap verified')
    return blocks


SPLITS = {}
print()
for H in HORIZONS:
    Xh, yh = build_horizon_dataset(hourly, H)
    SPLITS[H] = {'X': Xh, 'y': yh, 'blocks': split_report(Xh, yh, gap=H, name='H = ' + str(H) + ' h')}
    print()


split for H = 3 h (purge gap = 3 h):
            n_rows                from                  to  target_mean  target_std  target_max
block                                                                                          
train         6027 2024-01-02 00:00:00 2024-10-20 20:00:00        2.629       2.217      18.080
validation    1286 2024-10-21 03:00:00 2024-12-28 05:00:00        1.727       1.618      11.633
test          1290 2024-12-28 12:00:00 2025-03-08 03:00:00        2.235       2.388      16.783
  ordering and purge gap verified



C:\Users\Amir\AppData\Local\Temp\ipykernel_7288\1731899667.py:35: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  assert X.index[va[0]] - X.index[tr[-1]] >= pd.Timedelta(hours=gap), 'purge gap too small'
C:\Users\Amir\AppData\Local\Temp\ipykernel_7288\1731899667.py:35: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  assert X.index[va[0]] - X.index[tr[-1]] >= pd.Timedelta(hours=gap), 'purge gap too small'


split for H = 6 h (purge gap = 6 h):
            n_rows                from                  to  target_mean  target_std  target_max
block                                                                                          
train         6017 2024-01-02 00:00:00 2024-10-20 14:00:00        2.635       2.218      18.080
validation    1278 2024-10-21 03:00:00 2024-12-28 00:00:00        1.722       1.616      11.633
test          1286 2024-12-28 13:00:00 2025-03-08 03:00:00        2.229       2.382      16.783
  ordering and purge gap verified

split for H = 12 h (purge gap = 12 h):
            n_rows                from                  to  target_mean  target_std  target_max
block                                                                                          
train         5998 2024-01-02 00:00:00 2024-10-20 01:00:00        2.663       2.227      18.080
validation    1263 2024-10-21 02:00:00 2024-12-27 15:00:00        1.740       1.704      14.267
test          1277 2024-1

C:\Users\Amir\AppData\Local\Temp\ipykernel_7288\1731899667.py:35: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  assert X.index[va[0]] - X.index[tr[-1]] >= pd.Timedelta(hours=gap), 'purge gap too small'
C:\Users\Amir\AppData\Local\Temp\ipykernel_7288\1731899667.py:35: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  assert X.index[va[0]] - X.index[tr[-1]] >= pd.Timedelta(hours=gap), 'purge gap too small'


**Look closely at the `target_mean` and `target_std` rows.** They differ between blocks, and that is not a bug: the train block covers spring and summer, the validation and test blocks fall in autumn and winter. Consequences you must handle:

- A model tuned on the validation block may be slightly mis-tuned for the test block; prefer *robust* configurations over marginally better ones.
- Report the seasonal composition of each block in your methods section.
- Never 'fix' this by shuffling. That would trade an honest weakness for a silent lie.

**Check for understanding:** why is a three-block split (train/validation/test) preferable to cross-validation over the whole record here? *Because the test block must stay untouched during model selection. Time-aware cross-validation is used inside the training part only - Session 4 shows how, with `TimeSeriesSplit` and a gap.*

---

## 6. Scaling, and when you need it

| Model family | Needs scaling? | Why |
|---|---|---|
| Linear regression, ridge, lasso | yes (for regularised versions) | coefficients and penalties are scale-dependent |
| SVR, KNN, neural networks | **yes, mandatory** | distances dominate the objective |
| Decision trees, random forests, boosting | no | splits are invariant to monotone rescaling |

Rules for this course:

1. Fit the scaler on the **training block only**; transform validation and test with that same fitted scaler.
2. Use `sklearn.pipeline.Pipeline` so that scaling can never accidentally be fitted on the wrong data. This is not style - it is the mechanism that makes cross-validation leakage-safe: inside a pipeline, the scaler is refitted on each training fold.
3. Never scale the target unless you invert the prediction back to m/s. **A metric reported in scaled units is meaningless in a geoscience report.**

The demonstration below shows the pattern with a pipeline; Session 3 uses it for SVR and linear regression.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge


def linear_regression_available():
    '''Probe sklearn LinearRegression inside a throwaway subprocess.

    sklearn's LinearRegression delegates the fit to scipy.linalg.lstsq. On a few
    Windows conda builds that LAPACK call fast-fails and kills the Python process
    with no traceback at all - which would take the notebook kernel down with it.
    Probing in a subprocess turns a broken dependency into a printed message.
    '''
    import subprocess
    code = ('import numpy as np; from sklearn.linear_model import LinearRegression; '
            'X = np.random.RandomState(0).rand(20, 3); y = np.random.RandomState(1).rand(20); '
            'LinearRegression().fit(X, y)')
    try:
        probe = subprocess.run([sys.executable, '-c', code], capture_output=True, timeout=300)
        return probe.returncode == 0
    except Exception:
        return False


HAS_OLS = linear_regression_available()
if HAS_OLS:
    print('environment check: sklearn LinearRegression is usable')
else:
    print('[environment] sklearn LinearRegression is not usable here (scipy LAPACK fails).')
    print('              Falling back to Ridge(alpha=1e-6, solver=cholesky), which is')
    print('              numerically equivalent to ordinary least squares.')
    print('              Fix (optional): conda install -c conda-forge scipy --force-reinstall')

H_DEMO = 6
Xd, yd = SPLITS[H_DEMO]['X'], SPLITS[H_DEMO]['y']
tr, va, te = SPLITS[H_DEMO]['blocks']['train'], SPLITS[H_DEMO]['blocks']['validation'], SPLITS[H_DEMO]['blocks']['test']

# The pipeline refits the scaler on whatever data it is given, which is why it is
# safe inside cross-validation and why we must give it the training block only.
linear_estimator = LinearRegression() if HAS_OLS else Ridge(alpha=1e-6, solver='cholesky')
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', linear_estimator),
])
pipe.fit(Xd.iloc[tr], yd.iloc[tr])

for label, idx in [('train', tr), ('validation', va)]:
    pred = pipe.predict(Xd.iloc[idx])
    print('{:>10s}  MAE = {:.4f} m/s   R2 = {:+.4f}'.format(
        label, mean_absolute_error(yd.iloc[idx], pred), r2_score(yd.iloc[idx], pred)))

print()
print('Linear regression is only a weak sanity check here: the relationship between')
print('meteorological variables and future wind speed is strongly non-linear, so the')
print('interesting models come in Session 3. What matters now is the MECHANICS:')
print('scaler inside a pipeline, fitted on the training block, applied to the rest.')

sc = StandardScaler().fit(Xd.iloc[tr])
print()
print('example scaler statistics learned from the training block:')
show = ['wind_speed_ms', 'pressure_hpa', 'hour_sin']
print(pd.DataFrame({'mean_train': sc.mean_[[Xd.columns.get_loc(c) for c in show]],
                    'std_train': sc.scale_[[Xd.columns.get_loc(c) for c in show]]},
                   index=show).round(4).to_string())

environment check: sklearn LinearRegression is usable
     train  MAE = 1.1683 m/s   R2 = +0.3981
validation  MAE = 1.5218 m/s   R2 = -0.3462

Linear regression is only a weak sanity check here: the relationship between
meteorological variables and future wind speed is strongly non-linear, so the
interesting models come in Session 3. What matters now is the MECHANICS:
scaler inside a pipeline, fitted on the training block, applied to the rest.

example scaler statistics learned from the training block:
               mean_train  std_train
wind_speed_ms      2.6326     2.2383
pressure_hpa     759.0439     2.9097
hour_sin          -0.0001     0.7066


---

## 7. Feature selection

With ~93 features built from 6 variables, redundancy is guaranteed: `wind_speed_ms_roll24_mean` and `wind_speed_ms_lag24` measure nearly the same thing. Feature selection has three legitimate goals and one illegitimate one:

| Goal | Legitimate? |
|---|---|
| remove near-duplicate columns to cut training time | yes |
| remove columns that are constant or purely noise | yes |
| improve interpretability of the model | yes |
| find the columns that maximise the *test* score | **no - that is leakage** |

We apply three filters, **all computed on the training block only**:

1. **Variance filter** - drop columns with (near) zero variance in training.
2. **Mutual-correlation filter** - among pairs with `|r| > 0.98`, keep the one more correlated with the target.
3. **Correlation with the target** - inspect the ranking; use it as *understanding*, not as an automatic chopping rule.

Rule of thumb for tree ensembles: keep the redundancy and let the model handle it, unless training time or interpretation demands otherwise. For linear models and SVR, redundancy is genuinely harmful.

In [12]:
def select_features(X_train, y_train, min_std=1e-8, max_pair_corr=0.98, verbose=True):
    """Training-only feature filtering: variance, then near-duplicate pairs.

    Returns the list of retained column names.
    """
    # 1. variance filter
    keep = [c for c in X_train.columns if X_train[c].std() > min_std]
    dropped_var = [c for c in X_train.columns if c not in keep]

    # 2. near-duplicate pairs: computed with the Pearson correlation of the train block
    corr = X_train[keep].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    target_corr = X_train[keep].apply(lambda col: abs(np.corrcoef(col.values, y_train.values)[0, 1]))
    dropped_dup = []
    for col in upper.columns:
        partners = upper.index[upper[col] > max_pair_corr].tolist()
        for p in partners:
            weaker = col if target_corr[col] < target_corr[p] else p
            if weaker not in dropped_dup and weaker in keep:
                dropped_dup.append(weaker)
    keep = [c for c in keep if c not in dropped_dup]

    if verbose:
        print('features in                        :', X_train.shape[1])
        print('dropped by variance filter         :', len(dropped_var), dropped_var[:5])
        print('dropped as near-duplicates (r>0.98):', len(dropped_dup))
        print('  examples                         :', dropped_dup[:5])
        print('features out                       :', len(keep))
    return keep


FEATURES = {}
for H in HORIZONS:
    Xh, yh = SPLITS[H]['X'], SPLITS[H]['y']
    tr = SPLITS[H]['blocks']['train']
    print()
    print('--- H =', H, 'h ---')
    keep = select_features(Xh.iloc[tr], yh.iloc[tr])
    FEATURES[H] = keep


--- H = 3 h ---
features in                        : 93
dropped by variance filter         : 0 []
dropped as near-duplicates (r>0.98): 19
  examples                         : ['pressure_hpa_lag1', 'pressure_hpa_lag2', 'temperature_c_lag1', 'temperature_c_lag2', 'pressure_hpa_roll3_mean']
features out                       : 74

--- H = 6 h ---
features in                        : 93
dropped by variance filter         : 0 []
dropped as near-duplicates (r>0.98): 21
  examples                         : ['pressure_hpa', 'pressure_hpa_lag1', 'pressure_hpa_lag2', 'temperature_c', 'temperature_c_lag1']
features out                       : 72

--- H = 12 h ---
features in                        : 93
dropped by variance filter         : 0 []
dropped as near-duplicates (r>0.98): 21
  examples                         : ['pressure_hpa_lag1', 'pressure_hpa_lag2', 'pressure_hpa_lag3', 'temperature_c', 'temperature_c_lag2']
features out                       : 72

--- H = 24 h ---
features in       

In [13]:
# Which features actually carry information about the target?
# (computed on the TRAIN block; this is understanding, not automatic selection)
H_INSPECT = 6
Xh, yh = SPLITS[H_INSPECT]['X'], SPLITS[H_INSPECT]['y']
tr = SPLITS[H_INSPECT]['blocks']['train']

corr_target = Xh.iloc[tr].apply(lambda col: np.corrcoef(col.values, yh.iloc[tr].values)[0, 1])
ranked = corr_target.reindex(corr_target.abs().sort_values(ascending=False).index)
print('H = 6 h: linear correlation with the target (top 20, train block)')
print(ranked.head(20).round(3).to_string())
print()
print('weakest features:')
print(ranked.tail(5).round(4).to_string())

# optional mutual-information ranking (non-linear, slower, so subsample)
HAS_MI = False
try:
    from sklearn.feature_selection import mutual_info_regression
    HAS_MI = True
except Exception as exc:
    print()
    print('[optional] mutual_info_regression not available:', exc)

if HAS_MI:
    rng = np.random.RandomState(SEED)
    sub = rng.choice(len(tr), size=min(2000, len(tr)), replace=False)
    mi = mutual_info_regression(Xh.iloc[tr].iloc[sub], yh.iloc[tr].iloc[sub], random_state=SEED)
    mis = pd.Series(mi, index=Xh.columns).sort_values(ascending=False)
    print()
    print('top-10 features by mutual information with the target (non-linear):')
    print(mis.head(10).round(4).to_string())
    print()
    print('Compare the two rankings: features that are strong in BOTH linear')
    print('correlation and mutual information are the robust ones. A feature that is')
    print('strong only in mutual information is capturing a non-linear effect -')
    print('exactly what tree ensembles exploit.')

H = 6 h: linear correlation with the target (top 20, train block)
wind_speed_ms                0.420
wind_speed_ms_lag1           0.360
wind_speed_ms_roll24_mean    0.336
wind_speed_ms_roll3_mean     0.330
wind_speed_ms_roll3_max      0.315
wind_speed_ms_lag2           0.309
wind_speed_ms_roll6_mean     0.303
hour_sin                     0.297
wind_speed_ms_roll24_max     0.281
hour                        -0.279
wind_speed_ms_lag3           0.270
wind_speed_ms_roll6_max      0.268
wind_speed_ms_lag12          0.230
wind_speed_ms_lag6           0.209
wind_speed_ms_roll24_std     0.190
wind_dir_cos_roll6_mean      0.183
wind_dir_cos_lag6            0.156
wind_dir_cos_roll3_mean      0.155
wind_dir_sin_lag6            0.149
wind_dir_sin_lag12           0.147

weakest features:
wind_dir_cos_lag24        0.0115
month                     0.0071
pressure_hpa_roll3_std    0.0053
humidity_roll3_std       -0.0033
pressure_hpa_lag24        0.0012

top-10 features by mutual information with the ta

---

## 8. The final modelling dataset

The next cell assembles everything into one in-memory object, `MODEL_DATA`, which is all Sessions 3-7 need:

```
MODEL_DATA[H] = {
    'X_train', 'y_train',
    'X_val',   'y_val',
    'X_test',  'y_test',
    'features',          # retained column names
    'split_times',       # the datetime of every block
}
```

Design decisions baked into it (record them in your methods section):

1. hourly resolution, 10-minute records averaged;
2. sentinels and impossible values set to `NaN`, short gaps interpolated for auxiliary variables only;
3. direct multi-horizon targets at `H = 3, 6, 12, 24` h;
4. strictly-past rolling windows;
5. chronological 70/15/15 split with an `H`-hour purge gap;
6. near-duplicate features removed using training-only statistics;
7. the test block has not been looked at yet - and will not be, until Session 7.

In [14]:
MODEL_DATA = {}

for H in HORIZONS:
    Xh, yh = SPLITS[H]['X'], SPLITS[H]['y']
    cols = FEATURES[H]
    tr, va, te = SPLITS[H]['blocks']['train'], SPLITS[H]['blocks']['validation'], SPLITS[H]['blocks']['test']
    MODEL_DATA[H] = {
        'X_train': Xh.iloc[tr][cols].copy(),
        'y_train': yh.iloc[tr].copy(),
        'X_val': Xh.iloc[va][cols].copy(),
        'y_val': yh.iloc[va].copy(),
        'X_test': Xh.iloc[te][cols].copy(),
        'y_test': yh.iloc[te].copy(),
        'features': list(cols),
        'split_times': {'train': (Xh.index[tr[0]], Xh.index[tr[-1]]),
                        'validation': (Xh.index[va[0]], Xh.index[va[-1]]),
                        'test': (Xh.index[te[0]], Xh.index[te[-1]])},
    }

print('MODEL_DATA assembled for horizons', HORIZONS)
overview = []
for H in HORIZONS:
    d = MODEL_DATA[H]
    overview.append({
        'horizon_h': H,
        'features': len(d['features']),
        'train': len(d['y_train']),
        'val': len(d['y_val']),
        'test': len(d['y_test']),
        'train_mean_ws': d['y_train'].mean(),
        'test_mean_ws': d['y_test'].mean(),
    })
overview = pd.DataFrame(overview).set_index('horizon_h')
print(overview.round(3).to_string())

MODEL_DATA assembled for horizons [3, 6, 12, 24]
           features  train   val  test  train_mean_ws  test_mean_ws
horizon_h                                                          
3                74   6027  1286  1290          2.629         2.235
6                72   6017  1278  1286          2.635         2.229
12               72   5998  1263  1277          2.663         2.267
24               72   5963  1234  1260          2.707         2.277


In [15]:
# Final leakage audit of the assembled dataset - automatic where possible.
audit = []
for H in HORIZONS:
    d = MODEL_DATA[H]
    # (a) the training origin times must all precede the validation origin times
    ordered = (d['X_train'].index.max() < d['X_val'].index.min() < d['X_test'].index.min())
    # (b) the purge gap measured in hours
    gap_train_val = (d['X_val'].index.min() - d['X_train'].index.max()) / pd.Timedelta(hours=1)
    # (c) no feature may have been computed from the target column
    suspicious = [c for c in d['features'] if 'target' in c.lower()]
    audit.append({
        'horizon_h': H,
        'blocks_ordered': bool(ordered),
        'purge_gap_hours': float(gap_train_val),
        'gap_at_least_H': bool(gap_train_val >= H),
        'target_like_features': suspicious,
        'test_untouched': True,
    })
audit = pd.DataFrame(audit).set_index('horizon_h')
print(audit.to_string())

assert audit['blocks_ordered'].all(), 'split blocks are not ordered in time'
assert audit['gap_at_least_H'].all(), 'purge gap is smaller than the forecast horizon'
assert all(len(v) == 0 for v in audit['target_like_features']), 'a target-like feature leaked in'
print()
print('all automatic leakage checks passed')
print()
print('Remaining checks that CANNOT be automated - do them by reading the code:')
print('  1. every rolling window is built on a shifted (strictly past) series')
print('  2. no scaler/selector was fitted on data outside the training block')
print('  3. no bfill or bidirectional interpolation was used')
print('  4. the test block is opened for the first time in Session 7')

           blocks_ordered  purge_gap_hours  gap_at_least_H target_like_features  test_untouched
horizon_h                                                                                      
3                    True              7.0            True                   []            True
6                    True             13.0            True                   []            True
12                   True             25.0            True                   []            True
24                   True             49.0            True                   []            True

all automatic leakage checks passed

Remaining checks that CANNOT be automated - do them by reading the code:
  1. every rolling window is built on a shifted (strictly past) series
  2. no scaler/selector was fitted on data outside the training block
  3. no bfill or bidirectional interpolation was used
  4. the test block is opened for the first time in Session 7


C:\Users\Amir\AppData\Local\Temp\ipykernel_7288\723135394.py:8: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  gap_train_val = (d['X_val'].index.min() - d['X_train'].index.max()) / pd.Timedelta(hours=1)


---

## 9. Common pitfalls

**Pitfall 1 - the off-by-one in the target.**  
`shift(-H)` means the value at `T+H`. `shift(H)` means the value at `T-H`, i.e. pure history. Writing the wrong one silently turns the task into 'estimate the past', which any model does beautifully.

**Pitfall 2 - rolling windows that reach into the target.**  
`series.rolling(24).mean()` centred on `T+H`, or a target built as a rolling mean that includes the future, both encode the answer. Build every window from `series.shift(1)`.

**Pitfall 3 - shuffling.**  
Shown above: it inflates R2 by a large margin on autocorrelated data. `train_test_split(..., shuffle=True)` is the default in most tutorials, which is exactly why it is the most common mistake in student projects.

**Pitfall 4 - fitting the scaler (or the selector, or the imputer) on everything.**  
Use `Pipeline`. It is the cheapest insurance policy in machine learning.

**Pitfall 5 - interpolating the target.**  
A filled-in target teaches the model to reproduce your interpolation, not the atmosphere.

**Pitfall 6 - dropping rows with missing auxiliary data *before* the split.**  
If rows are dropped based on information from the whole record, the test block changes composition. Decide the cleaning rules on the training period and apply them everywhere.

**Pitfall 7 - confusing 'more features' with 'better model'.**  
With 6,000 training rows and 93 features, tree models can memorise. Compare train and validation scores explicitly (Session 3), and remember that a validation score can also be over-fitted by tuning too many times on it (Session 4).

**Pitfall 8 - seasonal drift.**  
Our blocks have different seasons. A model that looks weak on validation may be correct-but-cautious on a windy test winter, and vice versa. Report block composition and interpret differences in that light.

---

## 10. AI-assisted workflow

### 10.1 Prompts for feature engineering

| Goal | Prompt |
|---|---|
| generate a family of features | 'Write pandas code that, for an hourly wind-speed series, creates lags 1, 2, 3, 6, 12, 24 h and rolling means/max/std over the past 3, 6 and 24 h, using shift(1) before rolling so that the current hour is excluded from the window. Add a comment for each line explaining why the shift is needed.' |
| ask for the *hazards* instead of the code | 'List every way a rolling-window feature can leak future information in a 24-hour-ahead wind forecasting task. For each, give the wrong code and the fix in one line.' |
| cyclical encoding | 'Why is encoding the hour of day as sin/cos preferable to using the integer hour in a linear model? Give a numeric example with hours 23, 0 and 1.' |
| cross-check your design | 'Here is my feature list. For a forecast origin T and horizon H, mark each feature as safe, unsafe or ambiguous. Be strict.' |

### 10.2 Debugging prompts

```
My model gets R2 = 0.99 on a 24-hour-ahead wind speed forecast.
My features are: <list>.
My split is: <describe>.
List the five most likely causes, ordered by probability, and tell me the
fastest check that rules each one out. Do not suggest better models yet.
```

A high score is a bug report, not a success. Train the reflex.

### 10.3 Code-review prompt

```
Review this feature-engineering function as a strict reviewer for a geoscience
journal. For every line, state the timestamp of the data it touches and prove
that it is <= T. Flag anything you cannot prove. Then check the train/val/test
split for overlap, purging and scaling order. Output a table:
line | risk | severity | fix.
```

### 10.4 Validation checks for anything an LLM produces

1. **Trace one row by hand.** Pick a random origin `T`, print `X` and `y` for it, and check the target against the raw series at `T+H`. Do this for every new feature builder.
2. **Run the leaky-counterexample.** Ask the model to produce the *dishonest* version too, and confirm that it scores better. If it does not, your honest split is probably still leaking.
3. **Check that the API exists.** `shift`, `rolling`, `resample` and `interpolate` have subtle arguments - read the docstring in the notebook (`?df.rolling`).
4. **Beware of plausible statistics.** 'Use `zscore > 3` to remove outliers' - on a right-skewed physical variable this deletes real events. Ask what fraction of rows the rule removes and whether those rows are physically impossible or merely rare.
5. **Never accept a cleaning rule that changes the target distribution without understanding the mechanism.** Ask for the *effect* of the rule, then verify it on your data.

### 10.5 Exercise

Ask an LLM to write ‘a leakage-safe feature builder for hourly wind forecasting’. Then audit it line by line and list every leak you find. Most answers include at least one of: `fillna(method='bfill')`, a centred rolling window, a scaler fitted on the full data, or no purge gap.

---

## 11. Exercises

### Level 1 - understanding

1. State, in one sentence, the difference between `shift(1)` and `shift(-1)` in a feature-engineering context, and which one is legal for an input.
2. Why does `df.rolling(24).mean()` differ from `df.shift(1).rolling(24).mean()`, and which is safer?
3. What exactly does the purge gap remove, and what breaks if you set it to zero?
4. Give one reason why scaled targets are unacceptable in a geoscience report.

### Level 2 - implementation

5. Add a feature `wind_dir_change` = absolute angular change in direction over the past 3 hours, computed correctly and circularly. Verify its correlation with the target.
6. Add lag features for the *auxiliary* variables at 24 h. Does the validation MAE improve? By how much?
7. Build the dataset with `windows=(1,)` (i.e. almost no rolling statistics) and compare the number of features and the linear-regression validation MAE.
8. Replace the 70/15/15 split with 60/20/20 and compare the block composition in calendar terms.
9. Compute Spearman instead of Pearson correlation with the target and compare the top-10 rankings.

### Level 3 - reasoning

10. Explain why a model evaluated on a shuffled split reports a better score, and why that number is a *lower bound on the truth* rather than an upper bound.
11. The validation and test blocks are autumn/winter while the training block is spring/summer. List two ways this could change your conclusions, and how you would test for it.
12. Suppose an input sensor has an 8-hour outage. The auxiliary variables are interpolated over up to 2 hours. Describe what happens to the feature rows during the outage, and argue that this is the correct behaviour.
13. Your colleague removes all rows with wind speed above the IQR fence 'because they are outliers'. Predict the effect on (a) MAE, (b) R2, (c) the usefulness of the forecast. Is the model better?

### Level 4 - application (project work)

14. **Deliver the modelling dataset.** Run the assembly cell, then in a markdown cell document: number of features, number of rows per block, the block date ranges, the three cleaning decisions, and the four leakage checks you performed. This becomes the *Data and Methods* section of your report.
15. Add one **physically motivated** feature of your own - for example the pressure tendency over 3 hours (a classic synoptic predictor), or the difference between the station pressure and a 24 h mean. Compute it leakage-safely, report its correlation with the target and its mutual information, and say whether it is worth keeping.

---

## 12. Reproducibility notes

- **One source.** `data/raw/abali.xlsx`, read-only, cleaned in memory.
- **One seed.** `SEED = 42`, used for the random forest, the subsampling and the mutual-information estimate.
- **One recipe.** `build_horizon_dataset`, `chronological_split` and `select_features` are the only producers of the modelling table. Any change to them changes every result later - so if you modify them, record the change and re-run everything.
- **Frozen split.** The block boundaries are defined by row index, not by date strings, so they cannot drift between runs.
- **Nothing is written to disk.** Dataset assembly lives in memory; Sessions 3-7 rebuild it with the same code.
- **Two toggles you may set** and must then report: `QUICK_MODE` (smaller models; only affects runtime, never the split) and the hyper-parameters used in the demonstrations.
- **The test block is locked.** From this point it is not inspected, plotted or scored until Session 7. If you peek, say so in your report.

---

## 13. Key takeaways

1. A feature is legal if and only if it is computable at the forecast origin `T`. Write that sentence on the first page of your report.
2. The target for a direct multi-horizon model is `wind_speed.shift(-H)`; the last `H` rows are unusable.
3. Rolling windows are built from `shift(1)`, so they never include the current or a future hour.
4. Cyclical features (`sin`/`cos`) encode the hour and the day of year correctly; raw degrees for wind direction are never used as a feature.
5. Leakage was measured four times: target leakage, shuffling, back-filling and pre-split transformation. In our run, shuffling alone cut MAE by about 39 % and raised R2 by about 0.47, with no change to the model or the features.
6. The split is chronological with an `H`-hour purge gap; the test block is opened exactly once, in Session 7.
7. The modelling dataset for `H = 3, 6, 12, 24` h now exists in memory and is ready for modelling.

In [16]:
# ===========================================================================
# FINAL VALIDATION CELL - run last, in every notebook
# ===========================================================================
WARNINGS = []
if not HAS_SNS:
    WARNINGS.append('seaborn not installed: plots use matplotlib defaults')
if not HAS_MI:
    WARNINGS.append('mutual_info_regression unavailable: non-linear feature ranking skipped')
if QUICK_MODE:
    WARNINGS.append('QUICK_MODE=True: demonstration models use fewer trees ({} instead of 200)'.format(N_TREES))
WARNINGS.append('validation and test blocks fall in autumn/winter while training covers spring/summer: seasonal drift is expected')

manifest = {
    'notebook': '02_data_preparation_features_leakage_abali.ipynb',
    'session': 'Session 2 - data preparation, feature engineering and leakage',
    'dataset_used': RAW_PATH + ' (Abali station, hourly grid rebuilt in memory)',
    'target_variable': 'wind_speed_ms at T + H (m/s), direct multi-horizon',
    'forecast_horizons_hours': HORIZONS,
    'feature_families': ['current observations at T', 'lags (1,2,3,6,12,24 h)', 'past-only rolling mean/max/std (3,6,24 h)', 'calendar', 'cyclical sin/cos'],
    'feature_counts_per_horizon': {str(H): len(MODEL_DATA[H]['features']) for H in HORIZONS},
    'rows_per_block': {str(H): {'train': len(MODEL_DATA[H]['y_train']),
                                'validation': len(MODEL_DATA[H]['y_val']),
                                'test': len(MODEL_DATA[H]['y_test'])} for H in HORIZONS},
    'models_used': 'random forest (leakage demonstrations only), linear regression (scaling demonstration)',
    'metrics_used': ['MAE', 'R2 (demonstrations)'],
    'leakage_controls': ['strictly-past rolling windows via shift(1)',
                         'chronological split with purge gap = H',
                         'scalers and selectors fitted on the training block only',
                         'no bfill / no bidirectional interpolation',
                         'target never interpolated',
                         'test block not inspected in this session'],
    'files_written': 'none (all work in memory)',
    'random_seed': SEED,
    'warnings': WARNINGS,
}
print(json.dumps(manifest, indent=2, default=str))

{
  "notebook": "02_data_preparation_features_leakage_abali.ipynb",
  "session": "Session 2 - data preparation, feature engineering and leakage",
  "dataset_used": "../../data/raw/abali.xlsx (Abali station, hourly grid rebuilt in memory)",
  "target_variable": "wind_speed_ms at T + H (m/s), direct multi-horizon",
  "forecast_horizons_hours": [
    3,
    6,
    12,
    24
  ],
  "feature_families": [
    "current observations at T",
    "lags (1,2,3,6,12,24 h)",
    "past-only rolling mean/max/std (3,6,24 h)",
    "calendar",
    "cyclical sin/cos"
  ],
  "feature_counts_per_horizon": {
    "3": 74,
    "6": 72,
    "12": 72,
    "24": 72
  },
  "rows_per_block": {
    "3": {
      "train": 6027,
      "validation": 1286,
      "test": 1290
    },
    "6": {
      "train": 6017,
      "validation": 1278,
      "test": 1286
    },
    "12": {
      "train": 5998,
      "validation": 1263,
      "test": 1277
    },
    "24": {
      "train": 5963,
      "validation": 1234,
      "test": 

---

### What comes next

In **Session 3** the dataset built here gets its first real use:

- baselines first - persistence, climatology and diurnal climatology - because a model is only as good as the baseline it beats;
- then linear regression, a decision tree, a random forest and SVR, all on the same split;
- with an explicit discussion of underfitting and overfitting using train-versus-validation scores;
- and every model reported as a **skill score against persistence**.

Complete exercises 5, 10 and 15 before the next session; exercise 14 gives you the methods section of your final report.